On commence par importer les bibliothèques nécessaires. Attention : NE PAS EFFACER !

In [ ]:
TEST = False   # True : relecture de spectres sauvegardés (sans spectrophotomètre) / False : mesure réelle

Création du graphique

In [ ]:
# Graphique à deux volets : courbes des spectres (en haut), bandeau de couleurs (en bas) :
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.92, 0.08], vertical_spacing=0.02)

# Bandeau coloré : couleurs réelles du spectre visible (conversion λ → RGB) :
def couleurs_spectre(l):
    if l < 380 or l > 780:
        return (0, 0, 0)
    if l < 440:
        r, g, b = (440 - l) / 60, 0.0, 1.0
    elif l < 490:
        r, g, b = 0.0, (l - 440) / 50, 1.0
    elif l < 510:
        r, g, b = 0.0, 1.0, (510 - l) / 20
    elif l < 580:
        r, g, b = (l - 510) / 70, 1.0, 0.0
    elif l < 645:
        r, g, b = 1.0, (645 - l) / 65, 0.0
    else:
        r, g, b = 1.0, 0.0, 0.0
    if l < 420:
        f = 0.3 + 0.7 * (l - 380) / 40
    elif l > 700:
        f = 0.3 + 0.7 * (760 - l) / 60
    else:
        f = 1.0
    # les composantes sont déjà dans [0, 1], donc <= 255 après multiplication par 255
    return (int(f * r * 255), int(f * g * 255), int(f * b * 255))


longueurs = arange(350, 750, 1)
couleurs = [f"rgb({r},{g},{b})" for r, g, b in (couleurs_spectre(int(l)) for l in longueurs)]
fig.add_trace(go.Scatter(x=longueurs, y=zeros(len(longueurs)), mode="markers",
             marker=dict(color=couleurs, size=24, symbol="square"),
             showlegend=False, hoverinfo="skip"), row=2, col=1)    # bandeau sous l'axe des abscisses
fig.update_yaxes(showticklabels=False, showgrid=False, row=2, col=1)
fig.update_xaxes(title="Longueur d'onde (nm)", row=2, col=1)
# Le graphique n'est pas affiché ici : il apparaîtra avec le premier spectre (cellule suivante).

# Réglage du graphique (appliqués dès le premier spectre) :
#fig.update_yaxes(scaleanchor="x", scaleratio=1, row=1, col=1)    # à décommenter pour un repère orthonormé
fig.update_xaxes(showgrid=True, row=2, col=1)    # grille verticale (par défaut)
fig.update_yaxes(showgrid=True, row=1, col=1)    # grille horizontale (par défaut)
fig.update_layout(
    showlegend=True,    # légende du graphique
    title="Spectre UV visible",    # mettre le titre désiré entre les guillemets
    yaxis_title="Absorbance",    # nom de l'axe des ordonnées
    height=600)    # hauteur du graphique en pixels ; ex. : 450, 700...
print("Graphique prêt : les spectres s'ajouteront ici.")    # dernière ligne : rien n'est affiché ici

Lecture des données du spectrophotomètre  
Entrer un nouveau nom de colorant pour réaliser un nouveau spectre. Pour arrêter, valider sans entrer de valeur.  
Ordre et délai recommandés :  
· Idéal : **valider le nom puis lancer l’envoi** sur l’appareil aussitôt après (la lecture, déjà active, draine le buffer : rien ne se perd).  
· Envoyer puis valider : possible, le buffer absorbe ~10 s de flux ; attendre **moins de ~10 s**.  
· Viser au maximum **1 à 2 s** de délai pour rester large et sûr.

In [ ]:
while True :
    colorant = input("Entrer le nom du spectre :")
    if colorant == "":
        break

    import os
    os.makedirs("Spectres", exist_ok=True)
    fichier = f"Spectres/{colorant}.csv"

    if TEST:    # lecture sauvegardée (sans spectrophotomètre) : relecture d'un spectre
        if not os.path.exists(fichier):
            print(f"Fichier {fichier} introuvable : placer un spectre sauvegardé dans Spectres/")
            continue
        print(f"Lecture sauvegardée : {fichier}")
    else:
        donnees = ""
        precedente = None    # derniere longueur d'onde lue (pour detecter la fin d'un balayage)
        if not s.is_open:
            s.open()
        # NB : on ne vide PAS le buffer ici (avant la lecture) : après le
        # déclenchement de l'envoi, cela jetterait les premières longueurs d'onde
        # déjà émises (spectre tronqué au début). Le buffer est nettoyé APRÈS
        # chaque lecture (plus bas), quand l'appareil est silencieux : la lecture
        # suivante part donc toujours du vrai début de la salve.
        #
        # Ordre et délai recommandés (port 19200 bauds) :
        #  - Idéal : valider puis lancer l'envoi aussitôt après (lecture déjà
        #    active -> le buffer est drainé, rien ne se perd).
        #  - Envoyer puis valider : OK, le buffer absorbe ~10 s de flux (~20 Ko
        #    pour une salve de ~6 Ko émise en ~3 s). Attendre < ~10 s suffit.
        #  - Viser au maximum 1 à 2 s de délai pour rester large et sûr.

        # Déclencher la mesure sur le spectrophotomètre AVANT d'exécuter cette cellule.
        # On collecte les lignes de mesure jusqu'à 1 seconde de silence (fin de l'envoi) :
        fin = time() + 10    # durée maximale d'attente (en s)
        while time() < fin:
            ligne = s.readline()
            if ligne:
                texte = ligne.decode("utf8", errors="ignore")
                texte = re.sub(r"[^0-9.,\-\s]", "", texte)    # chiffres, points, virgules, signes, espaces
                texte = re.sub(r"[\t ]+", ",", texte)          # tabulations / espaces -> virgule (séparateur)
                texte = texte.replace("\r", "").strip()
                champs = texte.split(",")
                if len(champs) == 2:    # ligne de mesure : longueur d'onde, valeur
                    try:
                        lo = float(champs[0])
                        if 300 <= lo <= 1100:    # longueur d'onde plausible (nm)
                            # Un retour en arriere des longueurs d'onde (ex. 800 puis 400)
                            # signale le debut de la salve suivante : on arrete pour ne
                            # conserver qu'une seule salve complete.
                            if precedente is not None and lo < precedente:
                                break
                            donnees += texte + "\n"
                            precedente = lo
                            fin = time() + 1    # 1 s de silence -> acquisition terminee
                    except ValueError:
                        pass
        # Nettoyage APRÈS lecture (appareil silencieux) : vide les éventuels restes
        # pour que la prochaine lecture parte d'un buffer propre. Le port reste
        # ouvert (réutilisé d'une itération à l'autre).
        try:
            s.reset_input_buffer()
        except Exception:
            pass

        if not donnees.strip():    # aucune donnée reçue (mesure non déclenchée)
            print(f"Aucune donnée reçue pour {colorant} : vérifier le déclenchement de la mesure.")
            continue

        with open(fichier, "w") as f:    # ligne d'en-tête (noms des colonnes) puis données : l'en-tête du spectro est éliminé par la validation
            f.write("longueurs_ondes,A\n")
            f.write(donnees)

    tableurVersVariables(fichier)    # crée automatiquement les variables longueurs_ondes et A

    if TEST:
        print(f"Lecture sauvegardée : {len(longueurs_ondes)} points lus dans {fichier}")
    else:
        print(f"{len(longueurs_ondes)} points enregistrés pour {colorant}")

    # Ajout de la courbe du spectre au graphique, par-dessus les précédents
    # (si le même nom est re-saisi, sa courbe précédente est remplacée) :
    fig.data = [t for t in fig.data if t.name != colorant]
    fig.add_trace(go.Scatter(x=longueurs_ondes, y=A, mode="lines", name=colorant,
                             line=dict(width=2)), row=1, col=1)    # courbe dans le volet supérieur
    clear_output(wait=True)    # remplace l'affichage précédent (pas de graphique dupliqué)
    display(fig)               # graphique unique, mis à jour avec chaque nouveau spectre
print("Fin des enregistrements.")